# SAP ECC Treasury Data — Encoding, AI Analysis & Decoding
**Purpose:** Anonymize sensitive corporate data before sending to external AI for treasury insights.  
**Compliance principle:** No real entity names, account numbers, or document references leave the local environment.  
**Flow:** Clean data → Encode (anonymize) → Send to Claude API → Receive insights → Decode (restore real names) → Export for Power BI.

## 1. Setup & Load Clean Data

In [3]:
%pip install anthropic

import pandas as pd
import numpy as np
import json
import os
import hashlib
from datetime import datetime

# Load cleaned dataset
df = pd.read_csv('sap_ecc_treasury_clean.csv', encoding='utf-8-sig')
print(f'Loaded clean dataset: {len(df)} rows, {len(df.columns)} columns')
print(f'Columns: {list(df.columns)}')

Note: you may need to restart the kernel to use updated packages.
Loaded clean dataset: 10000 rows, 34 columns
Columns: ['MANDT', 'BUKRS', 'GJAHR', 'BELNR', 'BUZEI', 'BUDAT', 'BLDAT', 'VALUT', 'CPUDT', 'BLART', 'BSCHL', 'HKONT', 'SGTXT', 'WRBTR', 'DMBTR', 'WAERS', 'KURSF', 'LIFNR', 'KUNNR', 'NAME1', 'HBKID', 'HKTID', 'BANKN', 'RZAWE', 'DEAL_TYPE', 'DEAL_ID', 'DEAL_STATUS', 'MATURITY_DATE', 'COUNTERPARTY_BANK', 'PRCTR', 'KOSTL', 'ZUESSION', 'ERNAM', 'AEDAT']


## 2. Encoding Engine
The encoder replaces all identifiable corporate data with deterministic pseudonyms.  
**What gets encoded:** Entity names, account numbers, document references, company codes, user IDs.  
**What stays raw:** Amounts, dates, currencies, deal types, statuses, exchange rates — the AI needs these for analysis.  
**Amount scaling:** All monetary values are multiplied by a secret factor to hide the real financial scale.

In [4]:
class TreasuryDataEncoder:
    """
    Encodes corporate treasury data for safe external AI processing.
    Maintains a reversible mapping for decoding results.
    """
    
    def __init__(self, amount_scale_factor=None):
        # Deterministic but secret scaling factor for amounts
        self.amount_scale_factor = amount_scale_factor or round(np.random.uniform(0.6, 1.5), 4)
        
        # Mapping dictionaries (original → encoded)
        self.mappings = {
            'company_codes': {},
            'counterparty_names': {},
            'vendor_ids': {},
            'customer_ids': {},
            'bank_ids': {},
            'bank_accounts': {},
            'bank_numbers': {},
            'document_numbers': {},
            'deal_ids': {},
            'counterparty_banks': {},
            'cost_centers': {},
            'assignment_refs': {},
            'user_ids': {}
        }
        
        # Counters for sequential encoding
        self._counters = {k: 0 for k in self.mappings}
    
    def _get_code(self, category, prefix, original_value):
        """Get or create a deterministic encoded value."""
        if pd.isna(original_value) or str(original_value).strip() == '':
            return original_value
        
        original_value = str(original_value).strip()
        
        if original_value not in self.mappings[category]:
            self._counters[category] += 1
            self.mappings[category][original_value] = f'{prefix}_{self._counters[category]:03d}'
        
        return self.mappings[category][original_value]
    
    def encode(self, df):
        """Encode a DataFrame, returning the anonymized version."""
        df_enc = df.copy()
        
        # --- Company codes ---
        df_enc['BUKRS'] = df_enc['BUKRS'].apply(
            lambda x: self._get_code('company_codes', 'ENTITY', x))
        
        # --- Counterparty names ---
        df_enc['NAME1'] = df_enc['NAME1'].apply(
            lambda x: self._get_code('counterparty_names', 'CPTY', x))
        
        # --- Vendor IDs ---
        df_enc['LIFNR'] = df_enc['LIFNR'].apply(
            lambda x: self._get_code('vendor_ids', 'VND', x))
        
        # --- Customer IDs ---
        df_enc['KUNNR'] = df_enc['KUNNR'].apply(
            lambda x: self._get_code('customer_ids', 'CST', x))
        
        # --- House bank IDs ---
        df_enc['HBKID'] = df_enc['HBKID'].apply(
            lambda x: self._get_code('bank_ids', 'BNK', x))
        
        # --- Bank account IDs ---
        df_enc['HKTID'] = df_enc['HKTID'].apply(
            lambda x: self._get_code('bank_accounts', 'ACCT', x))
        
        # --- External bank account numbers ---
        df_enc['BANKN'] = df_enc['BANKN'].apply(
            lambda x: self._get_code('bank_numbers', 'IBAN', x))
        
        # --- Document numbers ---
        df_enc['BELNR'] = df_enc['BELNR'].apply(
            lambda x: self._get_code('document_numbers', 'DOC', x))
        
        # --- Deal IDs ---
        df_enc['DEAL_ID'] = df_enc['DEAL_ID'].apply(
            lambda x: self._get_code('deal_ids', 'DEAL', x))
        
        # --- Counterparty banks (FX deals) ---
        df_enc['COUNTERPARTY_BANK'] = df_enc['COUNTERPARTY_BANK'].apply(
            lambda x: self._get_code('counterparty_banks', 'FXBNK', x))
        
        # --- Cost centers ---
        df_enc['KOSTL'] = df_enc['KOSTL'].apply(
            lambda x: self._get_code('cost_centers', 'CC', x))
        
        # --- Assignment references (PO/Invoice numbers) ---
        df_enc['ZUESSION'] = df_enc['ZUESSION'].apply(
            lambda x: self._get_code('assignment_refs', 'REF', x))
        
        # --- User IDs ---
        df_enc['ERNAM'] = df_enc['ERNAM'].apply(
            lambda x: self._get_code('user_ids', 'USR', x))
        
        # --- Encode SGTXT (replace any real names that might appear in free text) ---
        for original, encoded in self.mappings['counterparty_names'].items():
            df_enc['SGTXT'] = df_enc['SGTXT'].str.replace(original, encoded, regex=False)
        for original, encoded in self.mappings['company_codes'].items():
            df_enc['SGTXT'] = df_enc['SGTXT'].str.replace(original, encoded, regex=False)
        
        # --- Scale amounts ---
        df_enc['WRBTR'] = (df_enc['WRBTR'] * self.amount_scale_factor).round(2)
        df_enc['DMBTR'] = (df_enc['DMBTR'] * self.amount_scale_factor).round(2)
        
        # --- Remove MANDT (client number is internal SAP config) ---
        if 'MANDT' in df_enc.columns:
            df_enc = df_enc.drop(columns=['MANDT'])
        
        return df_enc
    
    def save_mappings(self, filepath):
        """Save all mapping dictionaries + scale factor to JSON."""
        export = {
            'amount_scale_factor': self.amount_scale_factor,
            'encoded_at': datetime.now().isoformat(),
            'mappings': self.mappings
        }
        with open(filepath, 'w') as f:
            json.dump(export, f, indent=2)
        return filepath

print('Encoder class defined.')

Encoder class defined.


## 3. Run Encoding

In [5]:
# Initialize encoder
encoder = TreasuryDataEncoder(amount_scale_factor=0.7832)

# Encode the dataset
df_encoded = encoder.encode(df)

# Save mapping file locally (NEVER send this externally)
mapping_path = encoder.save_mappings('encoding_mappings.json')

# Save encoded dataset
df_encoded.to_csv('sap_ecc_treasury_encoded.csv', index=False, encoding='utf-8-sig')

print(f'Encoded dataset: {len(df_encoded)} rows, {len(df_encoded.columns)} columns')
print(f'Amount scale factor: {encoder.amount_scale_factor}')
print(f'Mapping file saved: {mapping_path}')
print(f'\nEncoding summary:')
for category, mapping in encoder.mappings.items():
    if mapping:
        print(f'  {category}: {len(mapping)} unique values encoded')

# Show sample: original vs encoded
print(f'\n--- SAMPLE: Original vs Encoded ---')
compare_cols = ['BUKRS', 'NAME1', 'LIFNR', 'HBKID', 'BELNR', 'WRBTR']
print(f'\nOriginal:')
print(df[compare_cols].head(3).to_string())
print(f'\nEncoded:')
print(df_encoded[compare_cols].head(3).to_string())

Encoded dataset: 10000 rows, 33 columns
Amount scale factor: 0.7832
Mapping file saved: encoding_mappings.json

Encoding summary:
  company_codes: 4 unique values encoded
  counterparty_names: 23 unique values encoded
  vendor_ids: 10 unique values encoded
  customer_ids: 8 unique values encoded
  bank_ids: 4 unique values encoded
  bank_accounts: 2 unique values encoded
  bank_numbers: 10000 unique values encoded
  document_numbers: 10000 unique values encoded
  deal_ids: 10000 unique values encoded
  counterparty_banks: 7 unique values encoded
  cost_centers: 16 unique values encoded
  assignment_refs: 6771 unique values encoded
  user_ids: 6 unique values encoded

--- SAMPLE: Original vs Encoded ---

Original:
   BUKRS                        NAME1 LIFNR HBKID       BELNR      WRBTR
0   1000           Axians Deutschland   NaN  CB01  5124005397  259919.34
1   3000  Cisco Systems International  V001  RB01  5124003077  243675.99
2   4000     Lenovo Global Technology  V005  RB01  5124006

## 4. Compliance Log
Document what was encoded, when, and what will be sent externally. Treasury/compliance teams expect this.

In [6]:
compliance_log = f"""
===============================================
DATA ANONYMIZATION COMPLIANCE LOG
===============================================
Timestamp: {datetime.now().isoformat()}
Source: sap_ecc_treasury_clean.csv
Output: sap_ecc_treasury_encoded.csv
Rows: {len(df_encoded)}

FIELDS ENCODED (PII/Corporate Identifiers):
  - BUKRS (Company Code) → ENTITY_XXX
  - NAME1 (Counterparty Name) → CPTY_XXX
  - LIFNR (Vendor ID) → VND_XXX
  - KUNNR (Customer ID) → CST_XXX
  - HBKID (House Bank ID) → BNK_XXX
  - HKTID (Bank Account ID) → ACCT_XXX
  - BANKN (External Bank Number) → IBAN_XXX
  - BELNR (Document Number) → DOC_XXX
  - DEAL_ID → DEAL_XXX
  - COUNTERPARTY_BANK → FXBNK_XXX
  - KOSTL (Cost Center) → CC_XXX
  - ZUESSION (Assignment Ref) → REF_XXX
  - ERNAM (User ID) → USR_XXX
  - SGTXT (Free Text) → Names replaced with encoded versions
  - WRBTR/DMBTR (Amounts) → Scaled by secret factor
  - MANDT (Client) → Removed entirely

FIELDS SENT RAW (Required for analysis):
  - GJAHR, BUDAT, BLDAT, VALUT, CPUDT, AEDAT (Dates)
  - WAERS (Currency Code)
  - KURSF (Exchange Rate)
  - BLART (Document Type)
  - BSCHL (Posting Key)
  - HKONT (G/L Account)
  - DEAL_TYPE, DEAL_STATUS
  - MATURITY_DATE
  - PRCTR (Profit Center)
  - RZAWE (Payment Method)
  - BUZEI (Line Item)

EXTERNAL AI SERVICE: Anthropic Claude API
MAPPING FILE: encoding_mappings.json (LOCAL ONLY — NOT transmitted)
AMOUNT SCALE FACTOR: Stored in mapping file (LOCAL ONLY)

CONFIRMATION: No real entity names, bank account numbers,
document references, or unscaled financial amounts were
transmitted to any external service.
===============================================
"""

with open('compliance_log.txt', 'w') as f:
    f.write(compliance_log)

print(compliance_log)


DATA ANONYMIZATION COMPLIANCE LOG
Timestamp: 2026-03-24T15:45:50.178849
Source: sap_ecc_treasury_clean.csv
Output: sap_ecc_treasury_encoded.csv
Rows: 10000

FIELDS ENCODED (PII/Corporate Identifiers):
  - BUKRS (Company Code) → ENTITY_XXX
  - NAME1 (Counterparty Name) → CPTY_XXX
  - LIFNR (Vendor ID) → VND_XXX
  - KUNNR (Customer ID) → CST_XXX
  - HBKID (House Bank ID) → BNK_XXX
  - HKTID (Bank Account ID) → ACCT_XXX
  - BANKN (External Bank Number) → IBAN_XXX
  - BELNR (Document Number) → DOC_XXX
  - DEAL_ID → DEAL_XXX
  - COUNTERPARTY_BANK → FXBNK_XXX
  - KOSTL (Cost Center) → CC_XXX
  - ZUESSION (Assignment Ref) → REF_XXX
  - ERNAM (User ID) → USR_XXX
  - SGTXT (Free Text) → Names replaced with encoded versions
  - WRBTR/DMBTR (Amounts) → Scaled by secret factor
  - MANDT (Client) → Removed entirely

FIELDS SENT RAW (Required for analysis):
  - GJAHR, BUDAT, BLDAT, VALUT, CPUDT, AEDAT (Dates)
  - WAERS (Currency Code)
  - KURSF (Exchange Rate)
  - BLART (Document Type)
  - BSCHL (P

## 5. Send to Claude API for Analysis
Send encoded data with a treasury-focused prompt. The AI analyzes patterns without ever seeing real corporate identifiers.  

**Before running this cell:**  
1. Install the Anthropic SDK: `pip install anthropic`  
2. Set your API key as an environment variable or paste it below.

In [ ]:
import anthropic
api_key = os.environ.get('ANTHROPIC_API_KEY', 'YOUR_KEY_HERE')
print('API key loaded successfully.')

API key loaded successfully.


In [8]:
# Prepare a statistical summary to send (not all 10K rows — that would exceed token limits)
# We send: summary statistics + a representative sample + aggregated views

# --- Summary stats ---
summary_stats = df_encoded.describe(include='all').to_string()

# --- Aggregated views treasury cares about ---
# Cash flow by entity and currency
cashflow_by_entity = df_encoded.groupby(['BUKRS', 'WAERS']).agg(
    total_amount=('WRBTR', 'sum'),
    transaction_count=('WRBTR', 'count'),
    avg_amount=('WRBTR', 'mean')
).round(2).to_string()

# FX exposure by currency
fx_deals = df_encoded[df_encoded['DEAL_TYPE'].isin(['FX_SPOT', 'FX_FORWARD'])]
fx_exposure = fx_deals.groupby(['WAERS', 'DEAL_TYPE', 'DEAL_STATUS']).agg(
    total_notional=('WRBTR', 'sum'),
    deal_count=('WRBTR', 'count'),
    avg_rate=('KURSF', 'mean')
).round(4).to_string()

# Payment method analysis
payment_analysis = df_encoded.groupby(['RZAWE', 'WAERS']).agg(
    total_amount=('WRBTR', 'sum'),
    count=('WRBTR', 'count')
).round(2).to_string()

# Monthly cash flow trend
df_encoded['month'] = pd.to_datetime(df_encoded['BUDAT']).dt.to_period('M').astype(str)
monthly_trend = df_encoded.groupby(['month', 'BLART']).agg(
    total_eur=('DMBTR', 'sum'),
    count=('DMBTR', 'count')
).round(2).to_string()

# Deal status overview
deal_overview = df_encoded.groupby(['DEAL_TYPE', 'DEAL_STATUS']).agg(
    total_amount=('WRBTR', 'sum'),
    count=('WRBTR', 'count')
).round(2).to_string()

# Top counterparties by volume
top_counterparties = df_encoded[df_encoded['NAME1'] != ''].groupby('NAME1').agg(
    total_amount=('WRBTR', 'sum'),
    transaction_count=('WRBTR', 'count')
).sort_values('total_amount', ascending=False).head(15).to_string()

# Sample rows (50 representative rows)
sample_rows = df_encoded.sample(50, random_state=42).to_string()

# Remove temp column
df_encoded = df_encoded.drop(columns=['month'], errors='ignore')

print('Data prepared for AI analysis.')
print(f'Aggregations: cashflow, FX exposure, payments, monthly trend, deals, counterparties')

Data prepared for AI analysis.
Aggregations: cashflow, FX exposure, payments, monthly trend, deals, counterparties


In [9]:
# Build the prompt
treasury_prompt = f"""You are a senior treasury analyst at a mid-size IT distribution company.
Analyze the following ANONYMIZED SAP ECC treasury data and provide actionable insights.

NOTE: All entity names are encoded (ENTITY_XXX, CPTY_XXX, etc.) and amounts are scaled for confidentiality.
Analyze patterns and ratios, not absolute values.

=== DATASET OVERVIEW ===
{summary_stats}

=== CASH FLOW BY ENTITY AND CURRENCY ===
{cashflow_by_entity}

=== FX EXPOSURE BREAKDOWN ===
{fx_exposure}

=== PAYMENT METHOD ANALYSIS ===
{payment_analysis}

=== MONTHLY CASH FLOW TREND ===
{monthly_trend}

=== DEAL STATUS OVERVIEW ===
{deal_overview}

=== TOP COUNTERPARTIES BY VOLUME ===
{top_counterparties}

=== SAMPLE TRANSACTIONS (50 rows) ===
{sample_rows}

Provide your analysis in the following structure. Use the encoded entity names (ENTITY_XXX, CPTY_XXX, etc.) as-is:

1. LIQUIDITY RISK ASSESSMENT
   - Net cash position patterns per entity
   - Cash flow timing mismatches (vendor payments vs customer receipts)
   - Short-term liquidity concerns (7/14/30 day windows)

2. FX EXPOSURE ANALYSIS
   - Currency concentration risk
   - Hedging ratio (forward deals vs spot deals)
   - Largest single-counterparty FX exposure
   - Recommendations for FX risk management

3. PAYMENT EFFICIENCY
   - Payment method cost analysis (wire vs SEPA vs ACH)
   - Opportunities to shift expensive wires to cheaper methods
   - Payment timing optimization

4. ANOMALY FLAGS
   - Unusually large transactions relative to entity averages
   - Counterparty concentration risk
   - Transactions that warrant manual review
   - Round-number transactions suggesting manual entries

5. CASH FLOW FORECASTING SIGNALS
   - Seasonal patterns (hardware vs software vs services revenue)
   - Vendor payment clustering patterns
   - Predictable inflows/outflows for better liquidity planning

6. KEY RECOMMENDATIONS (Top 5)
   - Ranked by potential financial impact
   - Each with estimated effort to implement
"""

print(f'Prompt prepared: ~{len(treasury_prompt.split())} words')
print(f'Sending to Claude API...')

Prompt prepared: ~3081 words
Sending to Claude API...


In [10]:
# Send to Claude API
client = anthropic.Anthropic(api_key=api_key)

response = client.messages.create(
    model='claude-sonnet-4-20250514',
    max_tokens=4096,
    messages=[
        {'role': 'user', 'content': treasury_prompt}
    ]
)

ai_response_encoded = response.content[0].text

# Save the raw encoded response
with open('ai_analysis_encoded.txt', 'w', encoding='utf-8') as f:
    f.write(ai_response_encoded)

print('AI analysis received and saved.')
print(f'Response length: {len(ai_response_encoded)} characters')
print(f'\n{"=" * 60}')
print('AI ANALYSIS (ENCODED — entity names still anonymized)')
print(f'{"=" * 60}')
print(ai_response_encoded)

AI analysis received and saved.
Response length: 7274 characters

AI ANALYSIS (ENCODED — entity names still anonymized)
# SAP ECC Treasury Data Analysis

## 1. LIQUIDITY RISK ASSESSMENT

**Net Cash Position Patterns:**
- **ENTITY_001** shows strongest liquidity with €592M total flows (4,994 transactions), averaging €118K per transaction
- **ENTITY_004** has concerning concentration in USD exposure (€272M vs €246M EUR), creating FX liquidity risk
- **ENTITY_003** shows lowest transaction volume (990 deals) but highest average transaction size (€152K), indicating lumpy cash flows

**Cash Flow Timing Mismatches:**
- Critical mismatch in FX deal settlement: 46% of FX forwards are still OPEN (722 deals worth €189M), while only 36% are settled
- Payment deals show similar pattern: 27% open (1,049 deals, €40M) vs 45% settled, indicating processing delays
- December shows spike in KZ transactions (€46M, 274 deals), suggesting year-end settlement rush

**Short-term Liquidity Concerns:**
- **HIG

## 6. Decode AI Response
Reverse the encoding: replace all pseudonyms with real entity names and rescale any amounts mentioned in the analysis.

In [11]:
class TreasuryDataDecoder:
    """
    Decodes AI response by reversing the encoding mappings.
    Replaces encoded identifiers with original corporate names.
    """
    
    def __init__(self, mapping_filepath):
        with open(mapping_filepath, 'r') as f:
            data = json.load(f)
        
        self.amount_scale_factor = data['amount_scale_factor']
        self.mappings = data['mappings']
        
        # Build reverse mappings (encoded → original)
        self.reverse_mappings = {}
        for category, mapping in self.mappings.items():
            for original, encoded in mapping.items():
                self.reverse_mappings[encoded] = original
    
    def decode_text(self, text):
        """Replace all encoded identifiers in text with originals."""
        decoded = text
        
        # Sort by length (longest first) to avoid partial replacements
        sorted_mappings = sorted(self.reverse_mappings.items(), 
                                  key=lambda x: len(x[0]), reverse=True)
        
        for encoded, original in sorted_mappings:
            decoded = decoded.replace(encoded, original)
        
        return decoded
    
    def decode_dataframe(self, df_encoded):
        """Reverse encoding on a DataFrame."""
        df_dec = df_encoded.copy()
        
        # Reverse map each encoded column
        reverse_by_category = {}
        for category, mapping in self.mappings.items():
            reverse_by_category[category] = {v: k for k, v in mapping.items()}
        
        col_category_map = {
            'BUKRS': 'company_codes',
            'NAME1': 'counterparty_names',
            'LIFNR': 'vendor_ids',
            'KUNNR': 'customer_ids',
            'HBKID': 'bank_ids',
            'HKTID': 'bank_accounts',
            'BANKN': 'bank_numbers',
            'BELNR': 'document_numbers',
            'DEAL_ID': 'deal_ids',
            'COUNTERPARTY_BANK': 'counterparty_banks',
            'KOSTL': 'cost_centers',
            'ZUESSION': 'assignment_refs',
            'ERNAM': 'user_ids'
        }
        
        for col, category in col_category_map.items():
            if col in df_dec.columns and category in reverse_by_category:
                df_dec[col] = df_dec[col].map(reverse_by_category[category]).fillna(df_dec[col])
        
        # Reverse amount scaling
        if self.amount_scale_factor != 0:
            df_dec['WRBTR'] = (df_dec['WRBTR'] / self.amount_scale_factor).round(2)
            df_dec['DMBTR'] = (df_dec['DMBTR'] / self.amount_scale_factor).round(2)
        
        return df_dec

print('Decoder class defined.')

Decoder class defined.


In [12]:
# Decode the AI response
decoder = TreasuryDataDecoder('encoding_mappings.json')

ai_response_decoded = decoder.decode_text(ai_response_encoded)

# Save decoded response
with open('ai_analysis_decoded.txt', 'w', encoding='utf-8') as f:
    f.write(ai_response_decoded)

print(f'{"=" * 60}')
print('AI ANALYSIS (DECODED — real entity names restored)')
print(f'{"=" * 60}')
print(ai_response_decoded)

AI ANALYSIS (DECODED — real entity names restored)
# SAP ECC Treasury Data Analysis

## 1. LIQUIDITY RISK ASSESSMENT

**Net Cash Position Patterns:**
- **1000** shows strongest liquidity with €592M total flows (4,994 transactions), averaging €118K per transaction
- **2000** has concerning concentration in USD exposure (€272M vs €246M EUR), creating FX liquidity risk
- **4000** shows lowest transaction volume (990 deals) but highest average transaction size (€152K), indicating lumpy cash flows

**Cash Flow Timing Mismatches:**
- Critical mismatch in FX deal settlement: 46% of FX forwards are still OPEN (722 deals worth €189M), while only 36% are settled
- Payment deals show similar pattern: 27% open (1,049 deals, €40M) vs 45% settled, indicating processing delays
- December shows spike in KZ transactions (€46M, 274 deals), suggesting year-end settlement rush

**Short-term Liquidity Concerns:**
- **HIGH RISK**: 357 FX_SPOT deals pending (€104M) - these typically settle T+2, creating imme

## 7. Structure Insights for Power BI
Parse the AI analysis into a structured format that Power BI can consume as a data source.

In [13]:
# Create structured insight records for Power BI
# Parse the AI response into categories

insight_categories = [
    'LIQUIDITY RISK ASSESSMENT',
    'FX EXPOSURE ANALYSIS',
    'PAYMENT EFFICIENCY',
    'ANOMALY FLAGS',
    'CASH FLOW FORECASTING SIGNALS',
    'KEY RECOMMENDATIONS'
]

# Split response by section headers
insights_records = []
sections = ai_response_decoded.split('\n')
current_category = 'GENERAL'
current_content = []

for line in sections:
    line_stripped = line.strip()
    
    # Check if this line is a category header
    matched_category = None
    for cat in insight_categories:
        if cat.lower() in line_stripped.lower():
            matched_category = cat
            break
    
    if matched_category:
        # Save previous section
        if current_content:
            insights_records.append({
                'category': current_category,
                'insight': '\n'.join(current_content).strip(),
                'generated_at': datetime.now().isoformat(),
                'source': 'Claude AI Analysis'
            })
        current_category = matched_category
        current_content = []
    else:
        if line_stripped:
            current_content.append(line_stripped)

# Don't forget the last section
if current_content:
    insights_records.append({
        'category': current_category,
        'insight': '\n'.join(current_content).strip(),
        'generated_at': datetime.now().isoformat(),
        'source': 'Claude AI Analysis'
    })

df_insights = pd.DataFrame(insights_records)
print(f'Parsed {len(df_insights)} insight sections:')
print(df_insights[['category']].to_string())

Parsed 7 insight sections:
                        category
0                        GENERAL
1      LIQUIDITY RISK ASSESSMENT
2           FX EXPOSURE ANALYSIS
3             PAYMENT EFFICIENCY
4                  ANOMALY FLAGS
5  CASH FLOW FORECASTING SIGNALS
6            KEY RECOMMENDATIONS


## 8. Export Everything for Power BI
Three files for Power BI import:  
1. **Clean dataset** — the main fact table  
2. **AI insights** — structured analysis table  
3. **Full decoded analysis** — text report

In [14]:
output_dir = os.path.dirname(os.path.abspath('__file__'))

# 1. Clean dataset (already exported in Phase 2, but ensure it's here)
clean_path = os.path.join(output_dir, 'sap_ecc_treasury_clean.csv')
print(f'Clean data: {clean_path}')

# 2. AI insights table
insights_path = os.path.join(output_dir, 'ai_treasury_insights.csv')
df_insights.to_csv(insights_path, index=False, encoding='utf-8-sig')
print(f'AI insights table: {insights_path}')

# 3. Also save as Excel for easier Power BI import
insights_xlsx = os.path.join(output_dir, 'ai_treasury_insights.xlsx')
df_insights.to_excel(insights_xlsx, index=False, engine='openpyxl')
print(f'AI insights Excel: {insights_xlsx}')

# 4. Full analysis text
analysis_path = os.path.join(output_dir, 'ai_analysis_decoded.txt')
print(f'Full analysis text: {analysis_path}')

# 5. Compliance log
print(f'Compliance log: compliance_log.txt')

print(f'\n{"=" * 60}')
print('PHASE 3 COMPLETE')
print(f'{"=" * 60}')
print(f'\nFiles ready for Power BI:')
print(f'  1. sap_ecc_treasury_clean.csv — Main fact table')
print(f'  2. ai_treasury_insights.csv — AI analysis (structured)')
print(f'  3. ai_analysis_decoded.txt — Full analysis report')
print(f'\nLocal-only files (DO NOT share):')
print(f'  - encoding_mappings.json — Decoding keys')
print(f'  - compliance_log.txt — Audit trail')
print(f'  - sap_ecc_treasury_encoded.csv — What was sent to AI')

Clean data: c:\Users\bobur\Downloads\sap_ecc_treasury_clean.csv
AI insights table: c:\Users\bobur\Downloads\ai_treasury_insights.csv
AI insights Excel: c:\Users\bobur\Downloads\ai_treasury_insights.xlsx
Full analysis text: c:\Users\bobur\Downloads\ai_analysis_decoded.txt
Compliance log: compliance_log.txt

PHASE 3 COMPLETE

Files ready for Power BI:
  1. sap_ecc_treasury_clean.csv — Main fact table
  2. ai_treasury_insights.csv — AI analysis (structured)
  3. ai_analysis_decoded.txt — Full analysis report

Local-only files (DO NOT share):
  - encoding_mappings.json — Decoding keys
  - compliance_log.txt — Audit trail
  - sap_ecc_treasury_encoded.csv — What was sent to AI
